## OpenAI Chat Responses Demo
Goal: basic interaction with GPT model

In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

print(api_key[:10])

sk-proj-4k


In [6]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

response = client.responses.create(
    model="gpt-4.1-mini",
    input="Who are you?"
)

print(response.output_text)

I am ChatGPT, an AI language model created by OpenAI. I'm here to help answer your questions, provide information, and assist with a wide range of topics. How can I assist you today?


In [7]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input="Tell me a joke"
)

print(response.output_text)

Sure! Here's a joke for you:

Why did the scarecrow win an award?  
Because he was outstanding in his field! 😄


In [9]:
response = client.responses.create(
    model="gpt-4.1-mini",
    instructions="""
    You are a senior software engineer.
    Write concise technical explanations.
    """,
    input="Write a summary of Python."
)

print(response.output_text)

Python is a high-level, interpreted programming language known for its readability and simplicity. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python features dynamic typing, automatic memory management, and a comprehensive standard library. It is widely used for web development, data science, automation, scripting, and scientific computing. Python's extensive ecosystem includes popular frameworks like Django and Flask, as well as data-focused libraries such as NumPy, pandas, and TensorFlow.


## Temperature Experiment

In [16]:
question = "Write a one-sentence bedtime story about a unicorn."

In [17]:
response = client.responses.create(
    model="gpt-4.1-mini",
    temperature=0,
    input=question
)

print(response.output_text)

Under a sky full of twinkling stars, a gentle unicorn named Luna whispered dreams of magic and kindness to every child as they drifted peacefully to sleep.


In [18]:
response = client.responses.create(
    model="gpt-4.1-mini",
    temperature=1,
    input=question
)

print(response.output_text)

Under a sky sprinkled with shimmering stars, a gentle unicorn named Luna whispered magical dreams to children as they softly drifted to sleep.


## Token Usage

In [14]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input="Explain transformers in simple terms."
)

print(response.usage)

ResponseUsage(input_tokens=13, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=258, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=271)


In [15]:
print("Input tokens:", response.usage.input_tokens)
print("Output tokens:", response.usage.output_tokens)
print("Total tokens:", response.usage.total_tokens)

Input tokens: 13
Output tokens: 258
Total tokens: 271


## Testing GPT-5.5

### Steps vs Outcome-first Prompting

In [21]:
instruction = "You are a senior AI solutions architect."

prompt_steps = """
You need to design a Telegram customer support bot for a SaaS company.

Requirements:
- Answer FAQ questions automatically.
- Escalate unresolved requests to human agents.
- Store conversation history.
- Support at least 1,000 conversations per day.

Constraints:
- Use Python.
- Keep infrastructure costs low.
- MVP must be delivered within 4 weeks.

Step 1: Analyze the requirements.
Step 2: Design the architecture.
Step 3: Create an implementation plan.
Step 4: Identify risks.
Step 5: Present your recommendations.
"""

prompt_outcome = """
Goal:
Design a Telegram customer support bot for a SaaS company.

Success criteria:
- Answer FAQ questions automatically.
- Escalate unresolved requests to human agents.
- Store conversation history.
- Support at least 1,000 conversations per day.

Constraints:
- Use Python.
- Keep infrastructure costs low.
- MVP must be delivered within 4 weeks.

Output:
- Recommended architecture
- Implementation plan
- Risks and mitigation strategies
"""

In [22]:
response = client.responses.create(
    model="gpt-5.5",
    instructions=instruction,
    input=prompt_steps
)

print(response.output_text)

## Step 1: Analyze the Requirements

### Functional Requirements

| Requirement | Interpretation |
|---|---|
| Answer FAQ questions automatically | The bot should understand user questions, search the FAQ/knowledge base, and respond with approved answers. |
| Escalate unresolved requests to human agents | If confidence is low, the customer asks for a human, or the issue is account-specific, the bot should create a human support handoff. |
| Store conversation history | Every message, bot response, escalation event, and resolution status should be persisted. |
| Support at least 1,000 conversations per day | The system should handle moderate traffic reliably, with room to scale. |

### Non-Functional Requirements

| Constraint | Impact |
|---|---|
| Python | Use Python-based bot framework and backend. |
| Low infrastructure cost | Prefer managed lightweight services or a small VPS. Avoid over-engineering. |
| MVP in 4 weeks | Use proven components, avoid complex custom AI pipelines, kee

In [23]:
response = client.responses.create(
    model="gpt-5.5",
    instructions=instruction,
    input=prompt_outcome
)

print(response.output_text)

## Recommended Architecture

### 1. High-Level Design

**Telegram User → Telegram Bot API → Python Bot Backend → FAQ/AI Engine → Database / Human Agents**

Core components:

1. **Telegram Bot**
   - Built with Python using either:
     - `aiogram` for async-first scalability, recommended
     - or `python-telegram-bot`
   - Uses Telegram webhooks instead of polling for production reliability.

2. **Backend API**
   - Python `FastAPI` service.
   - Receives Telegram webhook updates.
   - Handles:
     - User messages
     - FAQ matching
     - Escalation workflows
     - Conversation logging
     - Agent notifications

3. **FAQ Automation Layer**
   - MVP approach:
     - Start with curated FAQ knowledge base.
     - Use semantic search to find relevant answers.
   - Recommended stack:
     - OpenAI embeddings or a low-cost embedding model
     - Vector database using PostgreSQL with `pgvector`
   - Optional LLM layer:
     - Use an LLM to generate natural answers from approved FAQ cont

In [24]:
response = client.responses.create(
    model="gpt-5.5",
    instructions=instruction,
    input=prompt_outcome,
    text = {"verbosity": "low"},
    reasoning={'effort':'low'}
)

print(response.output_text)

## Recommended Architecture

### 1. High-level design

**Channels**
- Telegram Bot API via webhook

**Backend**
- Python service using **FastAPI**
- Telegram integration using **python-telegram-bot** or **aiogram**
- Hosted on low-cost infrastructure:
  - Render, Railway, Fly.io, Hetzner, DigitalOcean, or AWS Lightsail
  - Start with a single small VM/container

**Storage**
- **PostgreSQL** for:
  - Users
  - Conversations
  - Messages
  - Escalation tickets
  - Agent notes
- Optional: **Redis** for short-term session state and rate limiting  
  - Can be skipped for MVP if PostgreSQL is used carefully

**FAQ Automation**
- Store FAQ content in a knowledge base table or markdown files.
- Use one of two approaches:

#### MVP Option: Simple + Low Cost
- Keyword / semantic search over FAQ entries.
- Use embeddings for better matching:
  - OpenAI embeddings, Cohere, or local sentence-transformers.
  - Store vectors in PostgreSQL using **pgvector**.
- Generate or return answers based on matc